# Pulizia dati – Task 6 (Stile di vita salutare)

Pulisce i 3 dataset essenziali con age = TOTAL:
- Attività fisica
- Fumo
- Alcol

Esporta un CSV pulito.

In [32]:
import pandas as pd
from pathlib import Path

# >>> MODIFICA QUESTO PERCORSO <<<
CARTELLA_INPUT = Path("data")
CARTELLA_OUTPUT = Path("output")
CARTELLA_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Configurazione ok.")

Configurazione ok.


In [33]:
# ============================================================
# 1. ATTIVITÀ FISICA (CSV)
# ============================================================
df_pa = pd.read_csv(CARTELLA_INPUT / "T6.1_hlth_ehis_pe9e$defaultview_linear_2_0.csv")

df_pa = df_pa[
    (df_pa["isced11"] == "TOTAL") &
    (df_pa["age"] == "TOTAL") &
    (df_pa["physact"] == "MV_AERO")
].copy()

df_pa = df_pa[df_pa["TIME_PERIOD"] == df_pa["TIME_PERIOD"].max()]

df_pa = df_pa.rename(columns={
    "geo": "Paese",
    "sex": "Sesso",
    "OBS_VALUE": "Attivita fisica %"
})[["Paese", "Sesso", "Attivita fisica %"]].dropna()

print(f"Attività fisica: {len(df_pa)} righe")

Attività fisica: 96 righe


In [34]:
# ============================================================
# 2. FUMO (CSV)
# ============================================================
df_fumo = pd.read_csv(CARTELLA_INPUT / "T6.2_sdg_03_30__custom_24025495_linear_2_0.csv")

df_fumo = df_fumo[
    (df_fumo["age"] == "Y_GE15")
].copy()

df_fumo = df_fumo[df_fumo["TIME_PERIOD"] == df_fumo["TIME_PERIOD"].max()]

df_fumo = df_fumo.rename(columns={
    "geo": "Paese",
    "sex": "Sesso",
    "OBS_VALUE": "Fumo %"
})[["Paese", "Sesso", "Fumo %"]].dropna()

print(f"Fumo: {len(df_fumo)} righe")

Fumo: 84 righe


In [35]:
# ============================================================
# 3. ALCOL (CSV)
# ============================================================
df_alcol = pd.read_csv(CARTELLA_INPUT / "T6.3_hlth_ehis_al1e__custom_24027848_linear_2_0.csv")

df_alcol = df_alcol[
    (df_alcol["age"] == "TOTAL") &
    (df_alcol["frequenc"] == "DAY")
].copy()

df_alcol = df_alcol[df_alcol["TIME_PERIOD"] == df_alcol["TIME_PERIOD"].max()]

df_alcol = df_alcol.rename(columns={
    "geo": "Paese",
    "sex": "Sesso",
    "OBS_VALUE": "Alcol quotidiano %"
})[["Paese", "Sesso", "Alcol quotidiano %"]].dropna()

print(f"Alcol: {len(df_alcol)} righe")

Alcol: 84 righe


In [36]:
# ============================================================
# CONSOLIDAMENTO
# ============================================================
df = df_pa.merge(df_fumo, on=["Paese", "Sesso"], how="outer")
df = df.merge(df_alcol, on=["Paese", "Sesso"], how="outer")

# Mappa codici → nomi
NOMI_PAESI = {
    "AT": "Austria", "BE": "Belgio", "BG": "Bulgaria", "CY": "Cipro",
    "CZ": "Cechia", "DE": "Germania", "DK": "Danimarca", "EE": "Estonia",
    "EL": "Grecia", "ES": "Spagna", "FI": "Finlandia", "FR": "Francia",
    "HR": "Croazia", "HU": "Ungheria", "IE": "Irlanda", "IS": "Islanda",
    "IT": "Italia", "LT": "Lituania", "LU": "Lussemburgo", "LV": "Lettonia",
    "MT": "Malta", "NL": "Paesi Bassi", "NO": "Norvegia", "PL": "Polonia",
    "PT": "Portogallo", "RO": "Romania", "SE": "Svezia", "SI": "Slovenia",
    "SK": "Slovacchia", "TR": "Turchia", "RS": "Serbia"
}
df["Nome paese"] = df["Paese"].map(NOMI_PAESI).fillna(df["Paese"])

PAESI_UE = {
    "AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "EL", "ES", "FI",
    "FR", "HR", "HU", "IE", "IT", "LT", "LU", "LV", "MT", "NL", "PL",
    "PT", "RO", "SE", "SI", "SK"
}
df = df[df["Paese"].isin(PAESI_UE)].copy()

colonne_valori = ["Attivita fisica %", "Fumo %", "Alcol quotidiano %"]
for colonna in colonne_valori:
    df[colonna] = pd.to_numeric(df[colonna], errors="coerce")

# Score
df["Punteggio salute"] = (
    df["Attivita fisica %"].fillna(0)
    - df["Fumo %"].fillna(0)
    - df["Alcol quotidiano %"].fillna(0)
)
df["Punteggio salute"] = df["Punteggio salute"].round()

df = df.sort_values(["Sesso", "Punteggio salute"], ascending=[True, False])
df["Ranking"] = df.groupby("Sesso").cumcount() + 1

df = df[[
    "Ranking", "Paese", "Nome paese", "Sesso",
    "Attivita fisica %", "Fumo %", "Alcol quotidiano %", "Punteggio salute"
]]

df.to_csv(CARTELLA_OUTPUT / "stile_di_vita_pulito.csv", index=False)
print("Esportato: stile_di_vita_pulito.csv")
print(df.head(20).to_string(index=False))

Esportato: stile_di_vita_pulito.csv
 Ranking Paese  Nome paese Sesso  Attivita fisica %  Fumo %  Alcol quotidiano %  Punteggio salute
       1    NL Paesi Bassi     F               58.4    10.0                 6.7              42.0
       2    SE      Svezia     F               50.8     8.0                 1.2              42.0
       3    FI   Finlandia     F               53.8    15.0                 NaN              39.0
       4    DK   Danimarca     F               50.8    12.0                 6.8              32.0
       5    DE    Germania     F               45.2    20.0                 3.5              22.0
       6    IE     Irlanda     F               32.1    16.0                 1.4              15.0
       7    LU Lussemburgo     F               35.3    14.0                 6.2              15.0
       8    AT     Austria     F               41.4    25.0                 2.6              14.0
       9    SK  Slovacchia     F               24.6    17.0                 1.0   